In [12]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

In [13]:
full_merged_data = pd.read_csv(DATA_DIR / 'merge_hs_econ.csv')

In [14]:
# College attribute coverage (non-empty %) across the merged network dataset

college_attr_cols = [c for c in full_merged_data.columns if c.startswith('col_')]

print('Rows:', len(full_merged_data))
print('Unique colleges:', full_merged_data['college_id'].nunique())
print('College attribute columns:', len(college_attr_cols))
college_attr_cols

Rows: 1048575
Unique colleges: 2846
College attribute columns: 13


['col_name',
 'col_city',
 'col_st',
 'col_zip',
 'col_type',
 'col_ctyname',
 'col_ctyfips',
 'col_shparea',
 'col_shplength',
 'col_inst_control',
 'col_inst_size',
 'col_endow_total',
 'col_endow_per_fte']

In [15]:
def _non_empty_mask(s: pd.Series) -> pd.Series:
    if pd.api.types.is_string_dtype(s) or s.dtype == object:
        s_str = s.astype('string').str.strip()
        return s_str.notna() & (s_str != '')
    return s.notna()

summary_rows = []
n = len(full_merged_data)

for c in college_attr_cols:
    m = _non_empty_mask(full_merged_data[c])
    non_empty = int(m.sum())
    summary_rows.append({
        'field': c,
        'non_empty_cells': non_empty,
        'total_cells': n,
        'pct_non_empty': (non_empty / n * 100.0) if n else 0.0,
    })

college_attr_coverage = (
    pd.DataFrame(summary_rows)
      .sort_values('pct_non_empty', ascending=False)
      .reset_index(drop=True)
)
college_attr_coverage

,field,non_empty_cells,total_cells,pct_non_empty
0,col_name,1020156,1048575,97.289750
1,col_city,1020156,1048575,97.289750
2,col_st,1020156,1048575,97.289750
3,col_zip,1020156,1048575,97.289750
4,col_type,1020156,1048575,97.289750
5,col_ctyname,1020156,1048575,97.289750
6,col_ctyfips,1020156,1048575,97.289750
7,col_shparea,1020156,1048575,97.289750
8,col_shplength,1020156,1048575,97.289750
9,col_inst_control,885783,1048575,84.474930


In [16]:
# Colleges with NO data in any college attribute field (all col_* empty)

if not college_attr_cols:
    colleges_no_col_data = pd.DataFrame(columns=['college_id', 'n_rows', 'n_cycles'])
else:
    any_non_empty_by_row = pd.Series(False, index=full_merged_data.index)
    for c in college_attr_cols:
        any_non_empty_by_row |= _non_empty_mask(full_merged_data[c])

    any_non_empty_by_college = any_non_empty_by_row.groupby(full_merged_data['college_id']).any()
    missing_college_ids = any_non_empty_by_college[~any_non_empty_by_college].index

    colleges_no_col_data = (
        full_merged_data.loc[full_merged_data['college_id'].isin(missing_college_ids), ['college_id', 'cycle']]
          .groupby('college_id', as_index=False)
          .agg(n_rows=('college_id', 'size'), n_cycles=('cycle', 'nunique'))
          .sort_values(['n_rows', 'college_id'], ascending=[False, True])
          .reset_index(drop=True)
    )

print('Colleges with zero college-attribute matches:', len(colleges_no_col_data))
colleges_no_col_data.head()
colleges_no_col_data.to_csv(DATA_DIR / 'missing' / 'pub_col_no_data.csv', index=False)

Colleges with zero college-attribute matches: 140


In [17]:
# Public high school attribute coverage (non-empty %) across the merged network dataset

public_hs_df = full_merged_data.loc[full_merged_data['school_type'] == 'public'].copy()
hs_attr_cols = [c for c in full_merged_data.columns if c.startswith('hs_')]

print('Public HS rows:', len(public_hs_df))
print('Unique public HS ids:', public_hs_df['hs_id'].nunique())
print('HS attribute columns:', len(hs_attr_cols))
hs_attr_cols

Public HS rows: 803825
Unique public HS ids: 80285
HS attribute columns: 22


['hs_id',
 'hs_city',
 'hs_state',
 'hs_zip',
 'hs_ctyname',
 'hs_cty_fips',
 'hs_lat',
 'hs_long',
 'hs_pct_white',
 'hs_pct_black',
 'hs_pct_hispanic',
 'hs_pct_asian',
 'hs_pct_aian',
 'hs_pct_nhpi',
 'hs_pct_two_or_more',
 'hs_title_i_status',
 'hs_urban_centric_locale',
 'hs_charter',
 'hs_magnet',
 'hs_enrollment',
 'hs_pct_free_or_reduced_price_lunch',
 'hs_students_per_teacher']

In [18]:
summary_rows = []
n = len(public_hs_df)

for c in hs_attr_cols:
    m = _non_empty_mask(public_hs_df[c])
    non_empty = int(m.sum())
    summary_rows.append({
        'field': c,
        'non_empty_cells': non_empty,
        'total_cells': n,
        'pct_non_empty': (non_empty / n * 100.0) if n else 0.0,
    })

public_hs_attr_coverage = (
    pd.DataFrame(summary_rows)
      .sort_values('pct_non_empty', ascending=False)
      .reset_index(drop=True)
)
public_hs_attr_coverage

,field,non_empty_cells,total_cells,pct_non_empty
0,hs_id,803825,803825,100.000000
1,hs_charter,793149,803825,98.671850
2,hs_urban_centric_locale,793149,803825,98.671850
3,hs_state,782073,803825,97.293938
4,hs_zip,782073,803825,97.293938
5,hs_cty_fips,782073,803825,97.293938
6,hs_lat,782073,803825,97.293938
7,hs_long,782073,803825,97.293938
8,hs_city,782073,803825,97.293938
9,hs_enrollment,778848,803825,96.892732


In [19]:
# Public high schools with NO data in any hs_* field (excluding hs_id)

hs_attr_cols_no_id = [c for c in hs_attr_cols if c != 'hs_id']

if not hs_attr_cols_no_id:
    public_hs_no_hs_data = pd.DataFrame(columns=['hs_id', 'n_rows', 'n_cycles'])
else:
    any_non_empty_by_row = pd.Series(False, index=public_hs_df.index)
    for c in hs_attr_cols_no_id:
        any_non_empty_by_row |= _non_empty_mask(public_hs_df[c])

    any_non_empty_by_hs = any_non_empty_by_row.groupby(public_hs_df['hs_id']).any()
    missing_hs_ids = any_non_empty_by_hs[~any_non_empty_by_hs].index

    public_hs_no_hs_data = (
        public_hs_df.loc[public_hs_df['hs_id'].isin(missing_hs_ids), ['hs_id', 'cycle']]
          .groupby('hs_id', as_index=False)
          .agg(n_rows=('hs_id', 'size'), n_cycles=('cycle', 'nunique'))
          .sort_values(['n_rows', 'hs_id'], ascending=[False, True])
          .reset_index(drop=True)
    )

print('Public high schools with zero hs_* matches (excluding hs_id):', len(public_hs_no_hs_data))
public_hs_no_hs_data.head()
public_hs_no_hs_data.to_csv(DATA_DIR / 'missing' / 'pub_hs_no_data.csv', index=False)

Public high schools with zero hs_* matches (excluding hs_id): 396


In [20]:
# Investigate why certain public-HS fields have lower match rates

focus_cols = [
    'hs_students_per_teacher',
    'hs_pct_free_or_reduced_price_lunch',
    'hs_title_i_status',
    'hs_magnet',
]
focus_cols = [c for c in focus_cols if c in public_hs_df.columns]

cycle_counts = public_hs_df['cycle'].value_counts(dropna=False).sort_index()
cycle_counts

cycle
2019    210535
2020    122028
2021    157913
2022    180560
2023    132789
Name: count, dtype: int64

In [21]:
# % non-empty for focus fields by cycle (year)
if not focus_cols:
    pd.DataFrame({'note': ['None of the focus columns exist in public_hs_df']})
else:
    by_cycle = public_hs_df.groupby('cycle')[focus_cols].apply(
        lambda g: pd.Series({c: _non_empty_mask(g[c]).mean() * 100 for c in focus_cols})
    )
    by_cycle = by_cycle.round(2).sort_index()

    overall = pd.Series({c: _non_empty_mask(public_hs_df[c]).mean() * 100 for c in focus_cols}).round(2)
    print('Overall % non-empty (public HS):')
    display(overall)

    by_cycle

Overall % non-empty (public HS):


hs_students_per_teacher               89.87
hs_pct_free_or_reduced_price_lunch    84.60
hs_title_i_status                     59.89
hs_magnet                             59.89
dtype: float64

In [22]:
# Check whether missingness is clustered in certain hs_id's and whether fields co-occur
if not focus_cols:
    pd.DataFrame({'note': ['None of the focus columns exist in public_hs_df']})
else:
    # Co-occurrence for title_i / magnet (often sourced together)
    cols_pair = [c for c in ['hs_title_i_status', 'hs_magnet'] if c in public_hs_df.columns]
    if len(cols_pair) == 2:
        m_title = _non_empty_mask(public_hs_df[cols_pair[0]])
        m_magnet = _non_empty_mask(public_hs_df[cols_pair[1]])
        co = pd.crosstab(m_title, m_magnet, rownames=[cols_pair[0] + '_non_empty'], colnames=[cols_pair[1] + '_non_empty'])
        print('Co-occurrence (public HS rows):')
        display(co)

    # hs_id-level presence rate (share of cycles with any focus data)
    any_focus_by_row = pd.Series(False, index=public_hs_df.index)
    for c in focus_cols:
        any_focus_by_row |= _non_empty_mask(public_hs_df[c])

    hs_presence = (
        public_hs_df.assign(any_focus=any_focus_by_row)
          .groupby('hs_id', as_index=False)
          .agg(pct_cycles_with_any_focus=('any_focus', lambda s: s.mean() * 100),
               n_rows=('hs_id', 'size'),
               n_cycles=('cycle', 'nunique'))
          .sort_values(['pct_cycles_with_any_focus', 'n_rows'], ascending=[True, False])
          .reset_index(drop=True)
    )

    hs_presence.head(25)

Co-occurrence (public HS rows):


hs_magnet_non_empty,False,True
hs_title_i_status_non_empty,,
False,322448,0
True,0,481377


In [23]:
# Derived fields: see whether missingness aligns with missing/zero enrollment
out = {}

if 'hs_enrollment' in public_hs_df.columns and 'hs_students_per_teacher' in public_hs_df.columns:
    m_enroll = _non_empty_mask(public_hs_df['hs_enrollment'])
    m_ratio = _non_empty_mask(public_hs_df['hs_students_per_teacher'])
    out['pct_missing_students_per_teacher_given_enrollment_present'] = (
        (~m_ratio & m_enroll).mean() * 100
    )

if 'hs_enrollment' in public_hs_df.columns and 'hs_pct_free_or_reduced_price_lunch' in public_hs_df.columns:
    m_enroll = _non_empty_mask(public_hs_df['hs_enrollment'])
    m_pct = _non_empty_mask(public_hs_df['hs_pct_free_or_reduced_price_lunch'])
    out['pct_missing_lunch_pct_given_enrollment_present'] = (
        (~m_pct & m_enroll).mean() * 100
    )

pd.Series(out).round(2)

pct_missing_students_per_teacher_given_enrollment_present     7.02
pct_missing_lunch_pct_given_enrollment_present               12.29
dtype: float64